In [ ]:
!pip install faiss-cpu sentence-transformers transformers
!pip install -U langchain langchain-experimental langchain-text-splitters
!pip install -U langchain-huggingface

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import faiss
import numpy as np
import pandas as pd
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-small"
)

splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

# read marckdonw file
with open("نظام حماية البيانات الشخصية ولوائحه التنظيمية.md", "r", encoding="utf-8") as f:
    text = f.read()

# divide to Semantic Chunks
docs = splitter.create_documents([text])

#
documents = [doc.page_content for doc in docs]

print(f"Number of chunks: {len(documents)}")
print("\nFirst chunk:\n")
print(documents[0])

In [ ]:
embedding_model = SentenceTransformer("intfloat/multilingual-e5-small")
doc_embeddings = embedding_model.encode(
    documents,
    normalize_embeddings=True,
    convert_to_numpy=True
)

In [ ]:
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(doc_embeddings)

print(f"FAISS index contains {index.ntotal} documents.")

In [ ]:
def retrieve(query, top_k=2):
    query_emb = embedding_model.encode([query], normalize_embeddings=True)
    distances, indices = index.search(query_emb, top_k)
    retrieved_docs = [documents[i] for i in indices[0]]
    return retrieved_docs

In [ ]:
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    dtype="auto",
    device_map="auto",
)

In [ ]:
def rag_pipeline(query):
    retrieved_docs = retrieve(query)
    context = "\n\n".join(retrieved_docs)

    prompt = f"""You are an assistant for question answering.

Use ONLY the information provided in the context below.
If the answer is not found in the context, reply:
"I don't have enough information in the provided context."

Context:
{context}

Question:
{query}

Answer:
"""

    response = generator(
        prompt,
        max_new_tokens=150,
        temperature=0.3,
        do_sample=False
    )[0]["generated_text"]

    print("🔍 Retrieved Documents:\n", retrieved_docs)
    print("\n💬 Model Answer:\n", response)

In [ ]:
print(generator.model.device)

In [ ]:
rag_pipeline("ما المقصود بإخفاء الهوية في نظام حماية البيانات الشخصية")
